### Init

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from numpy.fft import rfft, irfft
from matplotlib.animation import FuncAnimation, PillowWriter
import fhd
import importlib
import time
import math
from pathlib import Path
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib as mpl
import pyogrio
from scipy.ndimage import distance_transform_edt
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata
from scipy.interpolate import RBFInterpolator
from scipy.optimize import lsq_linear
from shapely.geometry import box

importlib.reload(fhd)



In [ ]:
title_size = 16
ax_size = 14

species = ["low income", "middel income", "high income"]
species_colors = ["red", "green", "blue"]
bar_colors = ["darkred", "darkgreen", "darkblue"]

## Loading files

In [ ]:
DATA_DIR = Path("")

gdf_2011 = 

gdf_2012 = 

gdf_2013 =

gdf_2014 =

gdf_2015 =

gdf_2016 =

gdf_2017 =

gdf_2018 =

gdf_2019 =

gdf_2020 =

# cbsgebiedsindelingen2020.gpkg
area_path = DATA_DIR / "cbsgebiedsindelingen2020.gpkg"

layers = pyogrio.list_layers(area_path)

In [ ]:
gemeenten = gpd.read_file(
    area_path,
    layer="gemeente_niet_gegeneraliseerd"
)

corop = gpd.read_file(
    area_path,
    layer= "coropgebied_gegeneraliseerd"
)

# Replace invalid values with NaN
gdf_2020_clean = gdf_2020.copy()
num_cols = gdf_2020_clean.select_dtypes(include="number").columns
gdf_2020_clean[num_cols] = gdf_2020_clean[num_cols].mask(gdf_2020[num_cols] < -90000, np.nan)

gdf_2019_clean = gdf_2019.copy()
num_cols = gdf_2019_clean.select_dtypes(include="number").columns
gdf_2019_clean[num_cols] = gdf_2019_clean[num_cols].mask(gdf_2019[num_cols] < -90000, np.nan)

gdf_2018_clean = gdf_2018.copy()
num_cols = gdf_2018_clean.select_dtypes(include="number").columns
gdf_2018_clean[num_cols] = gdf_2018_clean[num_cols].mask(gdf_2018[num_cols] < -90000, np.nan)

gdf_2017_clean = gdf_2017.copy()
num_cols = gdf_2017_clean.select_dtypes(include="number").columns
gdf_2017_clean[num_cols] = gdf_2017_clean[num_cols].mask(gdf_2017[num_cols] < -90000, np.nan)

gdf_2016_clean = gdf_2016.copy()
num_cols = gdf_2016_clean.select_dtypes(include="number").columns
gdf_2016_clean[num_cols] = gdf_2016_clean[num_cols].mask(gdf_2016[num_cols] < -90000, np.nan)

gdf_2015_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2014_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2013_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2012_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

gdf_2011_clean = gdf_2015.copy()
num_cols = gdf_2015_clean.select_dtypes(include="number").columns
gdf_2015_clean[num_cols] = gdf_2015_clean[num_cols].mask(gdf_2015[num_cols] < -90000, np.nan)

## Crop to Area (The Hague-Rotterdam)

In [ ]:
# Selected areas
corop_names = [
    "Agglomeratie 's-Gravenhage",
    "Delft en Westland",
    "Groot-Rijnmond",
]

# First column = column-name in >2017 CBS data
# Second column = column_name in <2017 CBS data
# Third column = column_name in 2011 CBS data
# etc.
cols = [ 
    ("aantal_inwoners", "INWONER","INW2011", "INW2012", "INW2013", "INW2014", "Aantal inwoners"),
    ("percentage_laag_inkomen_huishouden", "P_LINK_HH","P_LINH2011", "P_LINH2012", "P_LINH2013", "P_LINH2014", "Laag inkomen"),
    ("percentage_midden_inkomen_huishouden", "P_MINK_HH","P_MIDDEL2011", "P_MIDDEL2012", "P_MIDDEL2013", "P_MIDDEL2014", "Midden inkomen"),
    ("percentage_hoog_inkomen_huishouden", "P_HINK_HH","P_HINH2011", "P_HINH2012", "P_HINH2013", "P_HINH2014", "Hoog inkomen"),
    ("aantal_woningen", "WONING"),
    ("aantal_niet_bewoonde_woningen", "WON_NBEW"),
    ("gemiddelde_woz_waarde_woning", "WOZWONING")
]

# Repeat for each year
dhr_gdf_reduced_2015, bounds_dhr, gemeenten_clipped_2015 = fhd.crop_and_rescale(
    year=2015,
    gdf=gdf_2015_clean,
    corop_gdf=corop,
    corop_names=corop_names,
    gemeenten = gemeenten
    cell_size=500,
    cols=cols
)

## Fill grid

In [ ]:
cell_size = 500 #Change to 100
n = int(bounds_dhr[2]-bounds_dhr[0]) // cell_size

ns = 3
N = (n, n)
Lx, Ly = (50,50)
dx = Lx/n
dy = Ly/n

x = np.arange(-Lx/2, Lx/2, dx)
y = np.arange(-Ly/2, Ly/2, dy)

minx, miny, maxx, maxy = bounds_dhr

centroids = dhr_gdf_reduced_2015.geometry.centroid

cols = ((centroids.x - minx) // cell_size).astype(int)
rows = ((centroids.y - miny) // cell_size).astype(int)

cols = np.clip(cols, 0, len(x)-1)
rows = np.clip(rows, 0, len(y)-1)

cols_inc = [ 
    ("percentage_laag_inkomen_huishouden", "P_LINK_HH","P_LINH2011", "P_LINH2012", "P_LINH2013", "P_LINH2014", "Laag inkomen"),
    ("percentage_midden_inkomen_huishouden", "P_MINK_HH","P_MIDDEL2011", "P_MIDDEL2012", "P_MIDDEL2013", "P_MIDDEL2014", "Midden inkomen"),
    ("percentage_hoog_inkomen_huishouden", "P_HINK_HH","P_HINH2011", "P_HINH2012", "P_HINH2013", "P_HINH2014", "Hoog inkomen"),
]

inx = 1 # Change for year to right column index
# Repeat for every year

# 2015
phi_rdh_2015 = np.full((ns, len(y), len(x)), np.nan)

# Fraction of each species
for i in range(ns):
    col_name = cols_inc[i][inx]
    phi_rdh_2015[i, rows, cols] = dhr_gdf_reduced_2015[col_name].fillna(np.nan) / 100


In [ ]:
cols_woz = ["WOZWONING", "gemiddelde_woz_waarde_woning"]

x = np.arange(-Lx/2, Lx/2, dx)
y = np.arange(-Ly/2, Ly/2, dy)

H_2016 = np.full((len(y), len(x)), np.nan)
H_2017 = np.full((len(y), len(x)), np.nan)
H_2018 = np.full((len(y), len(x)), np.nan)
H_2019 = np.full((len(y), len(x)), np.nan)
H_2020 = np.full((len(y), len(x)), np.nan)

centroids = dhr_gdf_reduced_2020.geometry.centroid

cols = ((centroids.x - minx) // cell_size).astype(int)
rows = ((centroids.y - miny) // cell_size).astype(int)

cols = np.clip(cols, 0, len(x)-1)
rows = np.clip(rows, 0, len(y)-1)

inx = 0

# 2016
col_name = cols_woz[inx]
H_2016[rows, cols] = dhr_gdf_reduced_2016[col_name].fillna(np.nan) / 1000

inx = 1
col_name = cols_woz[inx]

# 2017
H_2017[rows, cols] = dhr_gdf_reduced_2017[col_name].fillna(np.nan) / 1000

# 2018
H_2018[rows, cols] = dhr_gdf_reduced_2018[col_name].fillna(np.nan) / 1000

# 2019
H_2019[rows, cols] = dhr_gdf_reduced_2019[col_name].fillna(np.nan) / 1000

# 2020
H_2020[rows, cols] = dhr_gdf_reduced_2020[col_name].fillna(np.nan) / 1000

In [ ]:
# Repeat for each year

mirror_2015, nan_mask_2015 = fhd.mirrorred(phi_rdh_2015)
mirror_2016, nan_mask_2016 = fhd.mirrorred(phi_rdh_2016)
mirror_2017, nan_mask_2017 = fhd.mirrorred(phi_rdh_2017)
mirror_2018, nan_mask_2018 = fhd.mirrorred(phi_rdh_2018)
mirror_2019, nan_mask_2019 = fhd.mirrorred(phi_rdh_2019)
mirror_2020, nan_mask_2020 = fhd.mirrorred(phi_rdh_2020)

In [ ]:
H_2016, H_mask_2016 = fhd.mirrorred(H_2016)
H_2017, H_mask_2017 = fhd.mirrorred(H_2017)
H_2018, H_mask_2018 = fhd.mirrorred(H_2018)
H_2019, H_mask_2019 = fhd.mirrorred(H_2019)
H_2020, H_mask_2020 = fhd.mirrorred(H_2020)

# Model 1 
Linear utility

## Inference

In [ ]:
# Infere parameters for Delta_t is one year

D_t = 1
n_data = 5 # Number of years -1

# Add more years
mirror = [mirror_2015, mirror_2016, mirror_2017, mirror_2018, mirror_2019, mirror_2020]
masks = [nan_mask_2015, nan_mask_2016, nan_mask_2017, nan_mask_2018, nan_mask_2019, nan_mask_2020]
Thetas_M1_y1 = np.zeros((n_data, 3, 11))
std_M1_y1 = np.zeros((n_data, 3, 11))

for i in range(n_data):
    Thetas_M1_y1[i], std_M1_y1[i] = fhd.compute_thetas(
    mirror[i],
    mirror[i+1],
    masks[i],   # Mask from earliest year
    dx, dy,
    ns,
    delta_t = D_t,
    include_nu = False,
    include_Gamma= False,
)

In [ ]:
D_t = 2
n_data = 4
Thetas_M1_y2 = np.zeros((n_data, 3, 11))
std_M1_y2 = np.zeros((n_data, 3, 11))

for i in range(n_data):
    Thetas_M1_y2[i], std_M1_y2[i] = fhd.compute_thetas(
    mirror[i],
    mirror[i+2],
    masks[i],   # Mask from earliest year
    dx, dy,
    ns,
    delta_t = D_t,
    include_Gamma=False,
    include_nu = False
)

In [ ]:
D_t = 3
n_data = 3
Thetas_M1_y3 = np.zeros((n_data, 3, 11))
std_M1_y3 = np.zeros((n_data, 3, 11))

for i in range(n_data):
    Thetas_M1_y3[i], std_M1_y3[i] = fhd.compute_thetas(
    mirror[i],
    mirror[i+3],
    masks[i],   # Mask from earliest year
    dx, dy,
    ns,
    delta_t = D_t,
    include_Gamma=False,
    include_nu = False
)

In [ ]:
D_t = 4
n_data = 2
Thetas_M1_y4 = np.zeros((n_data, 3, 11))
std_M1_y4 = np.zeros((n_data, 3, 11))

for i in range(n_data):
    Thetas_M1_y4[i], std_M1_y4[i] = fhd.compute_thetas(
    mirror[i],
    mirror[i+4],
    masks[i],   # Mask from earliest year
    dx, dy,
    ns,
    delta_t = D_t,
    include_Gamma=False,
    include_nu = False
)

In [ ]:
D_t = 5
n_data = 1
Thetas_M1_y5 = np.zeros((n_data, 3, 11))
std_M1_y5 = np.zeros((n_data, 3, 11))

for i in range(n_data):
    Thetas_M1_y5[i], std_M1_y5[i] = fhd.compute_thetas(
    mirror[i],
    mirror[i+5],
    masks[i],   # Mask from earliest year
    dx, dy,
    ns,
    delta_t = D_t,
    include_Gamma=False,
    include_nu = False
)

## Parameter distribution

In [ ]:
thetas_all = {
    1: Thetas_M1_y1,
    2: Thetas_M1_y2,
    3: Thetas_M1_y3,
    4: Thetas_M1_y4,
    5: Thetas_M1_y5,
}

ns = 3
n_params = 11

grouped = {
    species: {
        param: np.concatenate([
            theta[:, species, param] / timestep
            for timestep, theta in thetas_all.items()
        ])
        for param in range(n_params)
    }
    for species in range(ns)
}

In [ ]:
# Ds_a = grouped[0][1] # Species, param

fig, ax  = plt.subplots(1,4, figsize= (20,5), sharey=True)

fig.suptitle("Parameter distribution for linear utility model" , size = title_size+2)

Ds = ['$D_a$', '$D_b$', '$D_c$']
stds = np.zeros(ns)
means = np.zeros(ns)

ax[0].axhline(0, color='lightgray')
for i in range(ns):
    params = grouped[i][0]
    stds[i] = np.std(params)
    means[i] = np.mean(params)
    x = np.array([Ds[i]] * len(params))
    ax[0].scatter(x, params, marker='o', alpha = 0.4, color = species_colors[i], label = species[i])
    ax[0].errorbar(Ds[i], means[i], yerr=[stds[i]], fmt='o', color=bar_colors[i], capsize=5)
ax[0].tick_params(axis='x', labelsize=ax_size)
ax[0].set_title("Diffusion coefficients", size = title_size)
ax[0].legend()

kappas_title = ['$\kappa_a$', '$\kappa_b$', '$\kappa_c$']

for sp in range(ns):
    ax[sp+1].axhline(0, color='lightgray')
    for i in range(ns):
        params = grouped[sp][i+1]
        stds[i] = np.std(params)
        means[i] = np.mean(params)
        x = np.array([kappas_title[i]] * len(params))
        # yerr = np.array([[0], [stds[i]]])
        ax[sp+1].scatter(x, params, marker='o', alpha = 0.4, color = species_colors[i], label = species[i])
        ax[sp+1].errorbar(kappas_title[i], means[i], yerr=[stds[i]], fmt='o', color=bar_colors[i], capsize=5)
    ax[sp+1].set_title(f"Interaction of {species[sp]}", size = title_size)
    ax[sp+1].tick_params(axis='x', labelsize=ax_size)




## Simulation

In [ ]:
Deltat = 1
param_M1_1y = []

for i in range(5):
    D_fit, kappa_fit, nu_fit, Gamma_fit = fhd.unpack_theta(Thetas_M1_y1[i], ns=3)

    Gamma_fit *= np.eye(ns)

    param_fitted = {
        'D': D_fit,
        'kappa': kappa_fit,
        'Gamma': Gamma_fit,
        'nu': nu_fit,
    }

    param_M1_1y.append(param_fitted)


In [ ]:
N = (n-1, n-1)
L = (50,50)
Lx, Ly = L
sim_2d_fitted = fhd.fhd_2d_3species(L,N, bc= 'Neumann', fft=False)

ns = sim_2d_fitted.nspecies

time_step = 1
dt = 1
nsteps = int(time_step / dt)
noise = 0
frames = 1

phi_M1_y1 = []

for i in range(5):
    st = time.time()
    phi_run = sim_2d_fitted.run(mirror[i], param_M1_1y[i], nsteps, dt, noise, frames, model = "Vitelli", verbatum = False)
    et = time.time()
    print(f"Simulation ran in t = {et-st:.6f} seconds")

    phi_masked = phi_run.copy()
    phi_masked[:,-1][masks[i+1]] = 0.0

    phi_M1_y1.append(phi_masked[:,-1])

## Performance

In [ ]:
n_years = 6

bases = np.zeros((n_years-1,ns))
mse_M1_y1 = np.zeros((n_years-1,ns))
bases_2y = np.zeros((n_years-1,ns))
mse_M1_y2 = np.zeros((n_years-2,ns))

for i, year in enumerate(range(2015, 2020)):
    data_year = eval(f"data_{year}")
    data_next_year = eval(f"data_{year+1}")
    bases[i] = [np.mean((data_year[sp,:] - data_next_year[sp,:])**2) for sp in range(ns)]

for i, year in enumerate(range(2016, 2021)):
    data_year = eval(f"data_{year}")
    mse_M1_y1[i] =  [np.mean((phi_M1_y1[i][sp,:] - data_year[sp,:])**2) / bases[i][sp] for sp in range(ns)]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
years = [2016, 2017, 2018, 2019, 2020]
ax.axhline(1, color='lightgray')
for sp in range(ns):
    plt.plot(years, mse_M1_y1[:,sp], marker='o', label=f'Linear utility ({species[sp]})', color = species_colors[sp], alpha = 0.7)
plt.xlabel('Year', size = ax_size)
plt.ylabel('Normalized MSE', size = ax_size)
plt.title('Normalized MSE of different models over time', size = title_size)
plt.xticks(years)
plt.legend()
plt.show()

# Model 2 (quadratic utility, gamma=0)

# Model 3 (quadratic utility, gamma!=0)